In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.blob import ContainerClient

TENANT_ID = "5a13decf-4ae8-423c-9d6f-320db1def1ad"
CLIENT_ID = "6fcdb218-bf19-483e-95ab-b86fb024790a"
secret = dbutils.secrets.get(catalog="workspace", schema="default", key="azure-reader-secret")

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, secret)
container = ContainerClient("https://stpowermarketfm.blob.core.windows.net", "raw", credential=credential)

In [0]:
names = [b.name for b in container.list_blobs(name_starts_with="weather/") if b.name.endswith(".json")]
print(len(names))
print(names[:2])

60
['weather/2026-10-09/weather_2026-10-09_NO1.json', 'weather/2026-10-09/weather_2026-10-09_NO2.json']


In [0]:
import json
raw = json.loads(container.download_blob(names[0]).readall())
ts = raw["properties"]["timeseries"]
print(len(ts))
print(ts[0])
print(ts[-1])

86
{'time': '2026-10-09T18:00:00Z', 'data': {'instant': {'details': {'air_pressure_at_sea_level': 999.5, 'air_temperature': 8.6, 'cloud_area_fraction': 99.9, 'relative_humidity': 61.3, 'wind_from_direction': 67.0, 'wind_speed': 2.4}}, 'next_12_hours': {'summary': {'symbol_code': 'lightrain'}, 'details': {}}, 'next_1_hours': {'summary': {'symbol_code': 'cloudy'}, 'details': {'precipitation_amount': 0.0}}, 'next_6_hours': {'summary': {'symbol_code': 'cloudy'}, 'details': {'precipitation_amount': 0.0}}}}
{'time': '2026-10-19T18:00:00Z', 'data': {'instant': {'details': {'air_pressure_at_sea_level': 1006.7, 'air_temperature': 10.9, 'cloud_area_fraction': 85.9, 'relative_humidity': 80.0, 'wind_from_direction': 245.9, 'wind_speed': 3.4}}}}


In [0]:
from datetime import datetime, timezone

rows = []
loaded_at = datetime.now(timezone.utc).isoformat()
for name in names:
    raw = json.loads(container.download_blob(name).readall())
    for t in raw["properties"]["timeseries"]:
        rows.append({
            "area": name[-8:-5],
            "time": t["time"],
            "data": json.dumps(t["data"]),
            "source_file": name,
            "loaded_at": loaded_at,
        })
print(len(rows))

5139


In [0]:
df = spark.createDataFrame(rows)
df.write.mode("overwrite").saveAsTable("workspace.default.bronze_weather")
print(spark.table("workspace.default.bronze_weather").count())

5139
